# Classification Part-A — Logistic Regression

This notebook implements one of the five required Classification Part-A algorithms using the Adult Income dataset.

## Problem Statement

Predict whether a person's annual income is **greater than $50K** from demographic, education, employment, and financial attributes.

# Classification Part-A — Dataset Audit and EDA

In [1]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "Review-1" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from classification_preprocessing import load_adult_dataset, prepare_features_target, build_preprocessor, save_result

RANDOM_STATE = 42

df = load_adult_dataset()

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())


Dataset shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  income          48842 non-null  object  
dtypes: category(8), int64(6), object(1)
memory usage: 3.0+ MB


None

## Dataset Audit

In [ ]:

print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nTarget distribution:")
display(df["income"].value_counts(dropna=False).to_frame("count"))


## EDA — Target Distribution

In [ ]:

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="income")
plt.title("Income Class Distribution")
plt.xlabel("Income Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## EDA — Numerical Features

In [ ]:

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[numeric_cols].describe().T)

df[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()


## EDA — Correlation Heatmap

In [ ]:

plt.figure(figsize=(10, 7))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Numerical Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


### EDA observations
After running the notebook, record the team's observations here. Discuss class balance, missing values, distributions, and notable relationships.

## Prepare Data

In [ ]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "Review-1" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from classification_preprocessing import load_adult_dataset, prepare_features_target, build_preprocessor, save_result

RANDOM_STATE = 42

df = load_adult_dataset()
X, y = prepare_features_target(df)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)
print("Positive class rate in training:", y_train.mean())
print("Positive class rate in testing :", y_test.mean())

preprocessor = build_preprocessor(X_train)


## Build and Train Model

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

In [ ]:

from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy       : {accuracy:.4f}")
print(f"Weighted F1    : {weighted_f1:.4f}")


## Confusion Matrix

In [ ]:

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["<=50K", ">50K"]
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
ax.set_title(f"Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.show()


## Result

In [ ]:

result = {
    "model": "Logistic Regression",
    "accuracy": accuracy,
    "weighted_f1": weighted_f1,
}

output_path = save_result(result, "01_logistic_regression.csv")
print("Saved:", output_path)
pd.DataFrame([result])


## Interpretation

Write the team's interpretation after running the model. Explain what the accuracy, weighted F1 score, and confusion matrix show.